# Walkthrough: regularization_pytorch_simple.ipynb

**Statistisches Lernen 2 — FH Kufstein Tirol**  
Arquivo de referência: `regularization_pytorch_simple.ipynb`

Este walkthrough explica passo a passo cada decisão do notebook original, como cada ferramenta funciona, e aponta as **diferenças críticas** em relação ao notebook mais completo (`mnist_regularization_pytorch.ipynb`) — incluindo um **bug real** no Early Stopping que precisa de atenção.

---

> **Estrutura de cada seção:** O que o código faz → Como a ferramenta funciona → Por que foi feito assim → Como proceder / adaptar

---

## Visão geral e diferenças em relação ao notebook complexo

Este notebook é uma versão **mais simples e interativa** do `mnist_regularization_pytorch.ipynb`. Em vez de rodar 8 experimentos em loop, você **configura um modelo por vez** mudando variáveis no topo da célula de treino.

| Característica | Simple (este) | Complexo (outro) |
|---------------|--------------|------------------|
| Camada linear | `nn.Linear` (padrão) | `ExplicitLinear` (customizada) |
| Quantos modelos | 1 por execução | 8 simultâneos em loop |
| Progress bar | `tqdm` ✓ | Sem tqdm |
| Hard Regularization | ✗ (não tem Max-Norm, clipping) | ✓ |
| L1/L2 aplica a | **Todos os parâmetros** (pesos + bias) | Apenas pesos |
| L2 calculado como | `torch.norm(p)` — norma L2 | `p.pow(2).sum()` — norma L2 ao quadrado |
| Conjunto de teste | 500 amostras (subset) | 10.000 amostras completo |
| Bug no Early Stopping | ⚠️ **Sim** — nunca ativa | ✗ (funciona corretamente) |

---

## Regularizações cobertas

| Tipo | Método | Como ativar |
|------|--------|-------------|
| Soft | **L2** | `lambda_l2 > 0` |
| Soft | **L1** | `lambda_l1 > 0` |
| Design | **Dropout** | `dropout > 0` no `MLP()` |
| Output | **Label Smoothing** | `label_smoothing > 0` |
| — | **Early Stopping** | `early_stopping = True` *(com bug, ver Seção 5)* |

---

## Seção 1 — Setup: `tqdm` e o restante dos imports

### O que o código faz

```python
from tqdm import tqdm
```

A única diferença nos imports em relação ao notebook complexo é a presença do `tqdm`.

---

### Como funciona: `tqdm`

`tqdm` é uma biblioteca de **progress bars** para loops Python. O nome vem do árabe *taqaddum* (تقدّم), que significa "progresso".

**Uso básico:** basta envolver qualquer iterável com `tqdm(iteravel)`:

```python
from tqdm import tqdm

# Sem tqdm:
for xb, yb in train_loader:
    ...

# Com tqdm — exibe barra de progresso:
for xb, yb in tqdm(train_loader):
    ...
```

**O que aparece no output:**
```
100%|██████████| 43/43 [00:07<00:00,  5.86it/s]
```

| Parte | Significado |
|-------|-------------|
| `100%` | Percentual completo |
| `43/43` | Batches processados / total de batches |
| `00:07` | Tempo decorrido |
| `00:00` | Tempo estimado restante |
| `5.86it/s` | Batches por segundo |

**Variantes úteis:**
```python
# Com descrição dinâmica:
pbar = tqdm(train_loader, desc=f'Epoch {epoch}')
for xb, yb in pbar:
    ...
    pbar.set_postfix({'loss': f'{loss.item():.4f}'})

# Para loops com range:
for epoch in tqdm(range(1, epochs+1), desc='Epochs'):
    ...
```

**Quando usar:** sempre que o loop levar mais de ~2 segundos. Evita o silêncio total durante treinamento longo.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from tqdm import tqdm
import random, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Demonstração do tqdm
print("\n--- Demonstração tqdm com range ---")
import time
for i in tqdm(range(5), desc='Simulando epochs'):
    time.sleep(0.1)

print("\n--- tqdm com postfix (mostra loss em tempo real) ---")
pbar = tqdm(range(5), desc='Treino')
for i in pbar:
    loss_fake = 1.0 / (i + 1)
    pbar.set_postfix({'loss': f'{loss_fake:.4f}', 'acc': f'{0.5 + i*0.1:.3f}'})
    time.sleep(0.1)
print("\ntqdm mostra informações úteis sem poluir o output com print()")

---

## Seção 2 — Data Pipeline: Diferenças em relação ao notebook complexo

### O que o código faz

```python
val_size = 500                          # antes era 1000
train_size = int(len(train_full)*0.1 - val_size)   # ~5500 (antes ~5000)

indices_test = torch.randperm(len(test_ds), ...)[:val_size]   # NOVO
small_test_ds = Subset(test_ds, indices_test)                  # NOVO

test_loader = DataLoader(small_test_ds, ...)   # apenas 500 amostras
```

Resultado: **train=5500 / val=500 / test=500**

---

### Diferença crítica: test set reduzido

O notebook complexo usava os **10.000 exemplos** do MNIST test set completo. Este notebook usa apenas **500**, criando um `Subset` aleatório do test set.

**Consequências:**
- A **test accuracy é muito mais ruidosa** — com 500 amostras, um erro em 5 amostras já muda a accuracy em 1%
- Comparações entre configurações são menos confiáveis
- Para fins educativos (rodar rápido, ver o efeito), é aceitável
- Para resultados reportáveis, **sempre use o test set completo**

**Erro padrão da accuracy com N amostras:**
$$\text{SE}(\hat{p}) = \sqrt{\frac{\hat{p}(1-\hat{p})}{N}}$$

Com $N=500$ e $\hat{p}=0.93$: SE ≈ 1.1% — cada resultado pode variar ±2% por acaso.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_full = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_ds    = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

val_size   = 500
train_size = int(len(train_full) * 0.1 - val_size)

indices      = torch.randperm(len(train_full), generator=torch.Generator().manual_seed(42))[:val_size + train_size]
indices_test = torch.randperm(len(test_ds),    generator=torch.Generator().manual_seed(42))[:val_size]

small_ds      = Subset(train_full, indices)
small_test_ds = Subset(test_ds, indices_test)

train_ds, val_ds = random_split(small_ds, [train_size, val_size],
                                 generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds,      batch_size=128, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,        batch_size=256, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(small_test_ds, batch_size=256, shuffle=False, num_workers=0, pin_memory=False)

print("=== Divisão dos dados ===")
print(f"  Train: {len(train_ds):,}  |  Val: {len(val_ds):,}  |  Test: {len(small_test_ds):,}")
print(f"  Batches de treino: {len(train_loader)} (batch_size=128)")

print()
print("--- Impacto do tamanho do test set na confiabilidade ---")
p_hat = 0.93
for N in [500, 1000, 5000, 10000]:
    se = np.sqrt(p_hat * (1 - p_hat) / N)
    print(f"  N={N:6d}: SE(accuracy) = ±{se:.3f} ({se*2*100:.1f}% de intervalo ±1.96σ)")

print()
print("Conclusão: com N=500, resultados variam ±2% por aleatoriedade — ruído alto.")
print("Para comparações sérias: use o test set completo (N=10000).")

---

## Seção 3 — Modelo: `nn.Linear` padrão vs `ExplicitLinear`

### O que o código faz

```python
class MLP(nn.Module):
    def __init__(self, hidden=64, dropout=0.0, use_bn=False):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28, hidden)     # ← padrão do PyTorch
        self.bn1 = nn.BatchNorm1d(hidden) if use_bn else None
        self.dropout = nn.Dropout(p=dropout) if dropout > 0 else None
        self.fc2 = nn.Linear(hidden, 10)         # ← padrão do PyTorch
```

---

### `nn.Linear` padrão: o que já vem incluído

`nn.Linear(in, out)` cria uma camada linear completa com:
- `layer.weight` — tensor de pesos shape `(out, in)`, Kaiming uniform por padrão
- `layer.bias` — tensor de bias shape `(out,)`
- Forward: `F.linear(x, self.weight, self.bias)` — idêntico ao `ExplicitLinear`

**Por que o notebook complexo usava `ExplicitLinear`?**  
Para mostrar explicitamente que os pesos são `nn.Parameter` e para aplicar Hard Regularization (`clamp_()`) diretamente no `.data`. Com `nn.Linear`, os pesos **já são** `nn.Parameter` — o comportamento é idêntico.

```python
layer = nn.Linear(10, 5)
print(type(layer.weight))   # <class 'torch.nn.parameter.Parameter'>
print(layer.weight.requires_grad)  # True — já é treinável
```

**Conclusão:** `nn.Linear` e `ExplicitLinear` são funcionalmente idênticos para treino normal. A versão explícita só é necessária quando você quer controle total sobre a inicialização ou constraints pós-step.

---

### Bloco de configuração: como usar o notebook

```python
model = MLP(hidden=64, dropout=0.0, use_bn=False).to(device)

epochs          = 10
label_smoothing = 0.0
lambda_l2       = 0.0
lambda_l1       = 1e-5
early_stopping  = False
patience        = 5
name            = 'model_L1'
```

O design do notebook é **configuração por variáveis**: você muda os valores nesta célula e re-executa. É diferente do notebook complexo que iterava sobre uma lista de dicts.

**Como ativar cada regularização:**

| Configuração | Valores para ativar |
|-------------|---------------------|
| Sem regularização (baseline) | `lambda_l1=0`, `lambda_l2=0`, `dropout=0.0`, `label_smoothing=0.0` |
| L1 | `lambda_l1=1e-5` (manter `lambda_l2=0`) |
| L2 | `lambda_l2=1e-4` (manter `lambda_l1=0`) |
| L1 + L2 (Elastic Net) | `lambda_l1=1e-6`, `lambda_l2=1e-4` |
| Dropout | `MLP(dropout=0.3)` |
| Label Smoothing | `label_smoothing=0.1` |
| Early Stopping | `early_stopping=True` *(corrigir o bug antes!)* |

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden=64, dropout=0.0, use_bn=False):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1     = nn.Linear(28*28, hidden)
        self.bn1     = nn.BatchNorm1d(hidden) if use_bn else None
        self.dropout = nn.Dropout(p=dropout)  if dropout > 0 else None
        self.fc2     = nn.Linear(hidden, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        if self.bn1     is not None: x = self.bn1(x)
        x = F.relu(x)
        if self.dropout is not None: x = self.dropout(x)
        x = self.fc2(x)
        return x

print("=== nn.Linear padrão ===")
layer = nn.Linear(10, 5)
print(f"  type(layer.weight): {type(layer.weight)}")
print(f"  layer.weight.shape: {layer.weight.shape}")
print(f"  layer.weight.requires_grad: {layer.weight.requires_grad}")
print(f"  layer.bias.shape: {layer.bias.shape}")

print()
print("=== Parâmetros do MLP ===")
set_seed(42)
mlp = MLP(hidden=64)
for nome, p in mlp.named_parameters():
    print(f"  {nome:20s}  shape={tuple(p.shape):15s}  numel={p.numel():,}")
total = sum(p.numel() for p in mlp.parameters())
print(f"  Total: {total:,} parâmetros")

print()
print("--- Diferença: for param in model.parameters() ---")
print("Itera TODOS os parâmetros (pesos E biases):")
for i, p in enumerate(mlp.parameters()):
    print(f"  param {i}: shape={tuple(p.shape)}  nome aproximado: {'peso' if p.dim()==2 else 'bias'}")

---

## Seção 4 — Training Loop: L1, L2 e suas diferenças sutis

### O que o código faz

```python
if lambda_l1 > 0:
    l1_reg = torch.tensor(0.)
    for param in model.parameters():
        l1_reg += torch.sum(torch.abs(param))
    loss = loss + lambda_l1 * l1_reg

if lambda_l2 > 0:
    l2_reg = torch.tensor(0.)
    for param in model.parameters():
        l2_reg += torch.norm(param)
    loss = loss + lambda_l2 * l2_reg
```

---

### Diferença 1: aplica em TODOS os parâmetros (pesos + bias)

```python
for param in model.parameters():   # ← pesos E biases
```

O notebook complexo aplicava L1 apenas nos pesos (`ExplicitLinear`), ignorando os biases. Este notebook aplica em **tudo** — pesos e biases.

**Por que regularizar só os pesos é mais comum?**
- Os biases são poucos parâmetros (64 + 10 = 74 vs 50.890 pesos)
- Regularizar biases raramente tem impacto significativo
- Mas **não é errado** regularizar tudo — apenas uma convenção

**Como separar pesos de biases:**
```python
# Regularizar só os pesos:
for nome, param in model.named_parameters():
    if 'weight' in nome:          # pula biases
        l1_reg += param.abs().sum()
```

---

### Diferença 2: `torch.norm(param)` vs `param.pow(2).sum()`

O notebook usa `torch.norm(param)` para L2, o que calcula a **norma L2** (raiz da soma dos quadrados):

$$\text{torch.norm}(\theta) = \|\theta\|_2 = \sqrt{\sum_j \theta_j^2}$$

A regularização L2 padrão usa a **norma ao quadrado**:

$$\text{L2 padrão} = \|\theta\|_2^2 = \sum_j \theta_j^2$$

**Qual a diferença prática?**

| Formulação | Derivada em relação a θⱼ | Efeito |
|------------|--------------------------|--------|
| $\lambda \|\theta\|^2$ (padrão) | $2\lambda\theta_j$ | Proporcional ao peso |
| $\lambda \|\theta\|$ (este notebook) | $\lambda \frac{\theta_j}{\|\theta\|}$ | Escala com a norma total |

Com `torch.norm`, o gradiente da regularização fica menor à medida que os pesos encolhem (porque $\|\theta\|$ diminui). Com a norma ao quadrado, o gradiente diminui linearmente com o peso.

**Para uso prático**, os dois funcionam como regularizadores — apenas o valor efetivo de $\lambda$ muda. Para equivalência com `weight_decay` do Adam, use a **norma ao quadrado**:

```python
# ✓ Equivalente a weight_decay no optimizer:
l2_reg += param.pow(2).sum()

# O que o notebook faz (diferente, mas funciona como regularizador):
l2_reg += torch.norm(param)
```

---

### `torch.tensor(0.)` — inicializar o acumulador

```python
l1_reg = torch.tensor(0.)
for param in model.parameters():
    l1_reg += torch.sum(torch.abs(param))
```

**Por que não usar `l1_reg = 0.0` (float Python)?**  
Porque `torch.sum(...).abs()` retorna um tensor PyTorch com `requires_grad=True`. Se você fizer `0.0 + tensor`, o resultado é um tensor — ok. Mas `torch.tensor(0.)` é mais explícito e garante que o acumulador começa como tensor PyTorch desde o início, ficando no grafo computacional.

**Forma mais idiomática (modern PyTorch):**
```python
# Mais limpo com sum() e generator expression:
l1_reg = sum(param.abs().sum() for param in model.parameters())
```

In [ ]:
# Demonstrar as diferenças entre as implementações de L1 e L2

set_seed(42)
mlp_demo = MLP(hidden=64).to(device)

print("=== Comparação L1: this notebook vs apenas pesos ===")
l1_todos = sum(param.abs().sum() for param in mlp_demo.parameters())
l1_pesos = sum(param.abs().sum() for nome, param in mlp_demo.named_parameters() if 'weight' in nome)
l1_bias  = sum(param.abs().sum() for nome, param in mlp_demo.named_parameters() if 'bias' in nome)
print(f"  L1 (todos os params):  {l1_todos.item():.2f}")
print(f"  L1 (só pesos):         {l1_pesos.item():.2f}")
print(f"  L1 (só biases):        {l1_bias.item():.4f}  ← muito pequeno relativamente")
print(f"  Diferença relativa: {(l1_todos - l1_pesos)/l1_todos * 100:.2f}%  (biases têm pouco impacto)")

print()
print("=== Comparação L2: torch.norm vs pow(2).sum() ===")
for nome, param in mlp_demo.named_parameters():
    if 'weight' not in nome: continue
    norm_l2     = torch.norm(param).item()
    norm_l2_sq  = param.pow(2).sum().item()
    print(f"  {nome:20s}: norm={norm_l2:.3f}  norm²={norm_l2_sq:.2f}  (norm²/norm = {norm_l2_sq/norm_l2:.2f} ≈ norm)")

print()
print("=== Gradiente: diferença na forma de penalização ===")
theta_demo = torch.tensor([3.0, 4.0], requires_grad=True)
print(f"  theta = {theta_demo.tolist()}  |  ||theta|| = {theta_demo.norm().item():.2f}")

# Gradiente de ||theta||^2 (L2 padrão)
loss_sq = theta_demo.pow(2).sum()
loss_sq.backward()
grad_sq = theta_demo.grad.clone()

# Gradiente de ||theta|| (o que o notebook usa)
theta_demo2 = torch.tensor([3.0, 4.0], requires_grad=True)
loss_norm = theta_demo2.norm()
loss_norm.backward()
grad_norm = theta_demo2.grad.clone()

print(f"  Gradiente de ||theta||²: {grad_sq.tolist()}  (= 2*theta — proporcional ao peso)")
print(f"  Gradiente de ||theta||:  {grad_norm.tolist()}  (= theta/||theta|| — vetor unitário)")
print("  Para λ=0.01: penalidade é similar, mas a dinâmica do gradiente difere.")

print()
print("=== torch.tensor(0.) vs 0.0 como acumulador ===")
# Com torch.tensor
reg1 = torch.tensor(0.)
for p in mlp_demo.parameters():
    reg1 = reg1 + p.abs().sum()
print(f"  Acumulador torch.tensor(0.): tipo={type(reg1).__name__}, requires_grad={reg1.requires_grad}")
# Forma moderna
reg2 = sum(p.abs().sum() for p in mlp_demo.parameters())
print(f"  Forma moderna sum(...):      tipo={type(reg2).__name__}, requires_grad={reg2.requires_grad}")
print(f"  São iguais? {torch.isclose(reg1, reg2)}")

---

## Seção 5 — ⚠️ BUG: Early Stopping nunca ativa

### O que o código faz (com bug)

```python
early_stopping = False   # variável Python booleana
patience       = 5

# ... dentro do loop:
if 'use_early_stopping' == True:   # ← BUG: compara STRING com booleano
    if val_loss < best_val - 1e-4:
        ...
```

---

### O bug: comparação string vs booleano

```python
'use_early_stopping' == True   # SEMPRE False — string nunca é igual a True
```

Em Python, uma string como `'use_early_stopping'` é **truthy** (toda string não-vazia é verdadeira em contexto booleano), mas **nunca é igual a** `True` com `==`.

```python
bool('use_early_stopping')          # True  — string não-vazia é truthy
'use_early_stopping' == True        # False — string ≠ booleano True
'use_early_stopping' is True        # False — objetos diferentes
```

**O que deveria ser:**
```python
if early_stopping:          # ✓ usa a variável booleana corretamente
    ...

# ou equivalentemente:
if early_stopping == True:  # ✓ também funciona
    ...
```

**Consequência do bug:** o Early Stopping **nunca para o treino** — o modelo sempre roda todos os `epochs`, ignorando a variável `patience` e `early_stopping=True`. O `best_state` nunca é salvo.

---

### Outro problema: `best_state` e `model.load_state_dict`

```python
if best_state is not None:
    model.load_state_dict(best_state)
```

Como o Early Stopping nunca ativa, `best_state` permanece `None` e esta linha nunca executa. O modelo final é simplesmente o estado após a última epoch — não necessariamente o melhor.

---

### Como corrigir

Substituir:
```python
if 'use_early_stopping'==True:
```

Por:
```python
if early_stopping:
```

E garantir que `best_state` seja salvo desde o início (primeiro epoch) para o `load_state_dict` final funcionar mesmo sem Early Stopping ativo:

```python
# Salvar o estado inicial (para que best_state nunca seja None)
best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
```

In [ ]:
# Demonstração do bug e da correção

print("=== Demonstração do BUG ===")
early_stopping = True   # variável que deveríamos checar

# O que o notebook faz (com bug):
resultado_bug = 'use_early_stopping' == True
print(f"  'use_early_stopping' == True  →  {resultado_bug}  (sempre False, BUG!)")

# Por que string não é True:
print(f"  bool('use_early_stopping') = {bool('use_early_stopping')}  (truthy, mas ≠ True)")
print(f"  'use_early_stopping' is True = {'use_early_stopping' is True}")
print(f"  'use_early_stopping' == True = {'use_early_stopping' == True}")

print()
print("=== Correção correta ===")
print(f"  early_stopping = {early_stopping}")
print(f"  if early_stopping:         →  {bool(early_stopping)}  ✓")
print(f"  if early_stopping == True: →  {early_stopping == True}  ✓")

print()
print("=== Outras armadilhas comuns de comparação em Python ===")
exemplos = [
    ("1 == True",        1 == True),
    ("0 == False",       0 == False),
    ("'' == False",      '' == False),
    ("[] == False",      [] == False),
    ("'texto' == True",  'texto' == True),
    ("bool('texto')",    bool('texto')),
    ("bool([])",         bool([])),
]
for expr, val in exemplos:
    print(f"  {expr:25s} →  {val}")

---

## Seção 6 — O Training Loop completo

### O que o código faz (visão completa)

```python
for epoch in range(1, epochs + 1):
    model.train()
    print('Starting epoch', epoch)
    for xb, yb in tqdm(train_loader):         # ← tqdm aqui
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss   = criterion(logits, yb)

        if lambda_l1 > 0:   # L1 manual
            l1_reg = torch.tensor(0.)
            for param in model.parameters():
                l1_reg += torch.sum(torch.abs(param))
            loss = loss + lambda_l1 * l1_reg

        if lambda_l2 > 0:   # L2 manual (norma)
            l2_reg = torch.tensor(0.)
            for param in model.parameters():
                l2_reg += torch.norm(param)
            loss = loss + lambda_l2 * l2_reg

        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
        n += xb.size(0)
```

---

### `loss.item()` — por que não usar o tensor diretamente

```python
running_loss += loss.item() * xb.size(0)
```

`.item()` extrai o valor escalar de um tensor de tamanho 1 como **float Python nativo**. Sem isso:
- O tensor permanece no grafo computacional
- Cada acumulação cria nodos novos no grafo → **memory leak** durante o loop
- Com muitas epochs, o programa pode consumir toda a RAM

```python
# ✗ Acumula tensores no grafo — memory leak:
running_loss += loss * xb.size(0)

# ✓ Extrai float, descarta o grafo:
running_loss += loss.item() * xb.size(0)
```

---

### `xb.size(0)` — por que multiplicar pelo batch size?

O `criterion` retorna a loss **média** do batch (mean reduction é o padrão). Para calcular a loss média correta sobre o epoch inteiro, precisamos desfazer essa média e acumular os totais:

```python
# A loss retornada é média: loss = SSE / batch_size
running_loss += loss.item() * xb.size(0)   # recuperar o total: média × n = soma
n            += xb.size(0)                  # contar amostras

train_loss = running_loss / n               # média correta sobre todo o epoch
```

Sem essa correção, o último batch (que pode ser menor) teria o mesmo peso que os outros, distorcendo a média.

---

### `model.eval()` na função `evaluate` — crítico

```python
def evaluate(model, loader, criterion):
    model.eval()                # ← OBRIGATÓRIO antes de avaliar
    ...
    with torch.no_grad():       # ← OBRIGATÓRIO para economizar memória
        ...
```

**`model.eval()`** desativa:
- **Dropout:** todos os neurônios ficam ativos (não zera nenhum)
- **BatchNorm:** usa a média/variância estatísticas calculadas durante treino, não as do batch atual

Esquecer `model.eval()` pode dar resultados inconsistentes — especialmente com Dropout alto, a accuracy de validação pode variar bastante entre chamadas.

**`model.train()`** no início de cada epoch reativa Dropout e BatchNorm em modo de treino.

In [ ]:
# Reproduzindo as funções de avaliação e o loop de treino

def accuracy(logits, y):
    return (logits.argmax(1) == y).float().mean().item()

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss   = criterion(logits, yb)
            total_loss    += loss.item() * xb.size(0)
            total_correct += (logits.argmax(1) == yb).sum().item()
            total         += xb.size(0)
    return total_loss / total, total_correct / total


# Demonstrar efeito de model.eval() com Dropout
print("=== Efeito de model.eval() vs model.train() com Dropout ===")
set_seed(42)
mlp_drop = MLP(dropout=0.5).to(device)
x_dummy  = torch.randn(1, 1, 28, 28).to(device)

mlp_drop.train()
outs_train = [mlp_drop(x_dummy).detach().cpu().numpy()[0] for _ in range(5)]

mlp_drop.eval()
outs_eval  = [mlp_drop(x_dummy).detach().cpu().numpy()[0] for _ in range(5)]

print("  Saídas em .train() (Dropout ativo — varia a cada call):")
for o in outs_train:
    print(f"    {o.round(2)}")

print("  Saídas em .eval() (Dropout desativo — sempre o mesmo):")
for o in outs_eval:
    print(f"    {o.round(2)}")

print()
print("=== Demonstração: loss.item() previne memory leak ===")
import tracemalloc
criterion_demo = nn.CrossEntropyLoss()
x_b = torch.randn(32, 784)
y_b = torch.randint(0, 10, (32,))
model_tiny = nn.Linear(784, 10)

# Simular acumulação
acc_sem_item = 0.0
acc_com_item = 0.0
for _ in range(100):
    logits = model_tiny(x_b)
    loss   = criterion_demo(logits, y_b)
    acc_com_item += loss.item()          # ✓ float puro
    # acc_sem_item += loss               # ✗ acumularia tensores

print(f"  Acumulador com .item(): tipo={type(acc_com_item).__name__}  valor={acc_com_item:.4f}")
print("  Sem .item(): criaria 100 nodos no grafo → memory leak em loops longos")

---

## Seção 7 — Training completo com configuração corrigida

Aqui treinamos o modelo com o Early Stopping **corrigido**, e demonstramos as diferentes configurações do notebook.

In [ ]:
def treinar_modelo(hidden=64, dropout=0.0, use_bn=False,
                   epochs=10, label_smoothing=0.0,
                   lambda_l2=0.0, lambda_l1=0.0,
                   early_stopping=False, patience=5,
                   nome='modelo', verbose=True):
    set_seed(42)
    model = MLP(hidden=hidden, dropout=dropout, use_bn=use_bn).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Salvar estado inicial para que best_state nunca seja None
    best_val   = float('inf')
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    wait       = 0
    history    = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    epoca_stop = epochs

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss, n = 0.0, 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss   = criterion(logits, yb)

            if lambda_l1 > 0:
                l1_reg = sum(p.abs().sum() for p in model.parameters())
                loss   = loss + lambda_l1 * l1_reg

            if lambda_l2 > 0:
                l2_reg = sum(torch.norm(p) for p in model.parameters())
                loss   = loss + lambda_l2 * l2_reg

            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
            n            += xb.size(0)

        train_loss = running_loss / n
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # Early Stopping CORRIGIDO: usar a variável, não a string
        if early_stopping:   # ← CORRIGIDO (era 'use_early_stopping' == True)
            if val_loss < best_val - 1e-4:
                best_val   = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                wait       = 0
            else:
                wait += 1
                if wait >= patience:
                    epoca_stop = epoch
                    if verbose: print(f'  Early stopping na epoch {epoch}')
                    break

    model.load_state_dict(best_state)
    _, test_acc = evaluate(model, test_loader, criterion)

    if verbose:
        print(f"  {nome}: val_loss={val_loss:.4f}  test_acc={test_acc:.4f}  (parou ep.{epoca_stop})")

    return model, history, test_acc


# Treinar configurações do notebook: baseline, L1, L2, Dropout, Label Smoothing
configs = [
    dict(nome='baseline',      lambda_l1=0.0,  lambda_l2=0.0),
    dict(nome='L1=1e-5',       lambda_l1=1e-5, lambda_l2=0.0),
    dict(nome='L2=1e-4',       lambda_l1=0.0,  lambda_l2=1e-4),
    dict(nome='dropout=0.3',   lambda_l1=0.0,  lambda_l2=0.0, dropout=0.3),
    dict(nome='LabelSmooth',   lambda_l1=0.0,  lambda_l2=0.0, label_smoothing=0.1),
    dict(nome='EarlyStopping', lambda_l1=0.0,  lambda_l2=0.0, early_stopping=True, patience=3),
]

resultados = []
for cfg in configs:
    m, h, acc = treinar_modelo(epochs=10, verbose=True, **cfg)
    resultados.append((cfg['nome'], h, acc, m))

# Plot das learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cores = plt.cm.tab10(np.linspace(0, 1, len(resultados)))

for (nome, h, acc, _), cor in zip(resultados, cores):
    axes[0].plot(h['val_loss'], color=cor, lw=2, label=nome)
    axes[1].plot(h['val_acc'],  color=cor, lw=2, label=nome)

axes[0].set_title('Val Loss por epoch'); axes[0].set_xlabel('Epoch')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].set_title('Val Accuracy por epoch'); axes[1].set_xlabel('Epoch')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.suptitle('Comparação das configurações — regularization_pytorch_simple', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Seção 8 — Visualizações: filtros, heatmap e histograma

As funções de visualização são **idênticas** às do notebook complexo — mesma lógica, mesmo código. A única diferença é que `n=64` na chamada `plot_fc1_filters` (mostra 64 filtros em vez de 16).

### Diferença: `n=64` — grid 8×8 de filtros

```python
plot_fc1_filters(model, name, n=64)   # ← 64 filtros, grid 8×8
```

O MLP tem 64 neurônios hidden, então `n=64` mostra **todos os filtros da primeira camada**. Com `k = int(math.sqrt(64)) = 8`, o grid é 8×8.

**Por que mostrar todos?** Com todos os filtros visíveis, é mais fácil comparar a **diversidade** dos padrões aprendidos:
- Baseline: filtros variados (bordas, curvas, padrões locais)
- L1: filtros mais esparsos (muitos próximos de zero, alguns muito ativos)
- Non-negativity: filtros apenas positivos (só ativação, não inibição)
- Dropout: filtros mais distribuídos (redundância forçada)

In [ ]:
# Reproduzindo as funções de visualização do notebook original

def plot_fc1_filters(model, name, n=16, cmap='coolwarm'):
    W    = model.fc1.weight.detach().cpu()
    vmax = float(W.abs().max())
    k    = int(math.sqrt(n))
    fig, axes = plt.subplots(k, k, figsize=(2*k, 2*k))
    for i, ax in enumerate(axes.flat[:n]):
        img = W[i].reshape(28, 28)
        im  = ax.imshow(img, cmap=cmap, vmin=-vmax, vmax=vmax)
        ax.set_title(f'{i}', fontsize=7)
        ax.axis('off')
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.5)
    plt.suptitle(f'fc1 — pesos como filtros 28×28  ({name})', fontsize=10)
    plt.tight_layout()
    plt.show()

def plot_fc2_heatmap(model, name, cmap='coolwarm'):
    W    = model.fc2.weight.detach().cpu()
    vmax = float(W.abs().max())
    plt.figure(figsize=(10, 3))
    plt.imshow(W, aspect='auto', cmap=cmap, vmin=-vmax, vmax=vmax)
    plt.colorbar(shrink=0.7)
    plt.yticks(range(10), [str(i) for i in range(10)])
    plt.xlabel('Hidden Neuron'); plt.ylabel('Classe')
    plt.title(f'fc2 — matriz de pesos  ({name})')
    plt.tight_layout()
    plt.show()

def plot_weight_hist(model, name, layer='fc1'):
    W = getattr(model, layer).weight.detach().cpu().view(-1)
    plt.figure(figsize=(7, 3))
    plt.hist(W.numpy(), bins=60, color='steelblue', alpha=0.85, edgecolor='white')
    plt.axvline(0, color='red', lw=1.5, ls='--')
    plt.title(f'Histograma de pesos {layer}  ({name})')
    plt.xlabel('Valor do peso'); plt.ylabel('Frequência')
    plt.tight_layout()
    plt.show()

def weight_stats(model):
    stats = {}
    for lname in ['fc1', 'fc2']:
        W = getattr(model, lname).weight.detach().cpu()
        stats[lname] = {
            'L1_norm':    float(W.abs().sum()),
            'L2_norm':    float(W.pow(2).sum().sqrt()),
            'max_abs':    float(W.abs().max()),
            'sparsity':   float((W.abs() < 1e-3).float().mean()),
        }
    return stats


# Visualizar baseline vs L1 (os dois mais distintos)
modelos_vis = {nome: m for nome, _, _, m in resultados if nome in ['baseline', 'L1=1e-5', 'dropout=0.3']}

for nome, mdl in modelos_vis.items():
    print(f"\n{'='*55}")
    print(f"Modelo: {nome}")
    plot_fc1_filters(mdl, nome, n=64)   # n=64: todos os filtros (igual ao original)
    plot_fc2_heatmap(mdl, nome)
    plot_weight_hist(mdl, nome, layer='fc1')
    st = weight_stats(mdl)
    print(f"  fc1 stats: L1={st['fc1']['L1_norm']:.1f}  L2={st['fc1']['L2_norm']:.2f}  "
          f"max={st['fc1']['max_abs']:.4f}  sparsity={st['fc1']['sparsity']:.4f}")

---

## Resumo das diferenças e armadilhas

### Diferenças em relação ao notebook complexo

| Ponto | Simple | Complexo | Impacto |
|-------|--------|----------|---------|
| Camada linear | `nn.Linear` | `ExplicitLinear` | Nenhum no treino; complexo tem acesso direto `.data` |
| L1/L2 aplica em | Todos os params (pesos + bias) | Só pesos | L1/L2 norm ligeiramente maior aqui |
| L2 fórmula | `torch.norm(p)` (norma) | `p.pow(2).sum()` (norma²) | Dinâmica do gradiente diferente; λ precisa de ajuste |
| Test set | 500 amostras (ruidoso) | 10.000 (confiável) | Accuracy aqui tem ±2% de incerteza |
| Progress bar | `tqdm` | Sem | Conforto de uso; sem impacto funcional |
| Early Stopping | **Bug: nunca ativa** | Funciona | Early stopping aqui é inoperante sem correção |
| Hard constraints | ✗ | ✓ (Max-Norm, clipping) | Funcionalidade ausente aqui |

### Armadilhas documentadas

1. **`'use_early_stopping' == True`** → sempre `False` — corrigir para `if early_stopping:`
2. **`torch.norm(p)` para L2** → não é a formulação padrão — usar `p.pow(2).sum()` para equivalência com `weight_decay`
3. **Test set com N=500** → alta variância estatística — não usar para comparações reportáveis
4. **L1/L2 sobre biases** → convenção menos comum — geralmente regulariza-se só os pesos

---

## Guia de ferramentas: referência rápida

| Ferramenta | Sintaxe | O que faz |
|------------|---------|----------|
| Progress bar | `for x in tqdm(iteravel, desc=...)` | Barra de progresso com velocidade e ETA |
| Postfix tqdm | `pbar.set_postfix({'loss': f'{l:.4f}'})` | Exibe métricas ao lado da barra |
| Extrai float de tensor | `tensor.item()` | Evita acumulação de grafo e memory leak |
| Tamanho do batch | `xb.size(0)` ou `len(xb)` | Número de amostras no batch atual |
| L1 sobre todos params | `sum(p.abs().sum() for p in model.parameters())` | Soma total dos valores absolutos |
| L2 como norma | `sum(torch.norm(p) for p in model.parameters())` | Norma L2 (não ao quadrado) |
| L2 padrão (quadrado) | `sum(p.pow(2).sum() for p in model.parameters())` | Equivalente ao weight_decay |
| Só pesos, sem biases | `for n, p in model.named_parameters() if 'weight' in n` | Filtra parâmetros por nome |
| Modo treino | `model.train()` | Ativa Dropout e BatchNorm em modo treino |
| Modo avaliação | `model.eval()` | Desativa Dropout, usa stats fixas BatchNorm |
| Salvar estado | `model.state_dict()` | Dicionário com todos os pesos |
| Carregar estado | `model.load_state_dict(state)` | Restaura pesos de um checkpoint |

---

## Como adaptar este notebook para seus próprios experimentos

**Passo 1 — Escolher a configuração:**
```python
model = MLP(hidden=64, dropout=0.0, use_bn=False).to(device)
epochs          = 15          # mais epochs para datasets maiores
label_smoothing = 0.0         # 0.1 para classificação com ruído
lambda_l2       = 0.0         # 1e-4 para regularização L2
lambda_l1       = 0.0         # 1e-5 para L1 (sparsity)
early_stopping  = True        # sempre ativar após corrigir o bug
patience        = 5           # 3-10 epochs típico
```

**Passo 2 — Interpretar as curvas:**
```
val_loss > train_loss (gap grande) → overfitting → aumentar lambda
val_loss ≈ train_loss (ambos altos) → underfitting → reduzir lambda ou mais epochs
val_loss sobe após certa epoch → Early Stopping vai pegar o melhor ponto
```

**Passo 3 — Interpretar os filtros:**
```
Filtros nítidos e variados → modelo aprendeu features distintas
Filtros todos parecidos → neurônios colineares (considerar Dropout)
Histograma com cauda longa → pesos grandes → considerar L2
Histograma estreito perto de zero → L1 funcionou, sparsity alta
```

---

## Exercícios para consolidar

1. **Corrigir o bug:** troque `'use_early_stopping' == True` por `if early_stopping:` e observe quando o treino para agora.
2. **Elastic Net:** ative `lambda_l1=1e-6` e `lambda_l2=1e-4` simultaneamente. Os filtros ficam mais esparsos ou menos?
3. **Comparar L2 norm vs L2 squared:** implemente `p.pow(2).sum()` no lugar de `torch.norm(p)`. Precisa de λ diferente para atingir o mesmo efeito?
4. **Regularizar só os pesos:** modifique o loop L1 para usar `named_parameters()` e pular os biases. A diferença é mensurável com N=500?
5. **tqdm com postfix:** adicione `pbar.set_postfix({'loss': f'{loss.item():.4f}'})` dentro do loop de batches. O que aparece na tela?
6. **test set completo:** mude `small_test_ds` para usar `test_ds` completo (10.000 amostras). Os resultados ficam mais estáveis ao rodar duas vezes?